In [1]:
from __future__ import annotations

import json
import logging
import re
import subprocess
import types
from pathlib import Path

import pandas as pd
import soccerdata as sd
import soccerdata._common as common
import undetected_chromedriver as uc
import yaml
from pymongo import MongoClient

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "dev_scripts" else Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
CHROME_PATH = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("season_lineups")

print(PROJECT_ROOT)

[07/25/26 11:51:56] INFO     No custom team name replacements found. You can configure these in       ]8;id=15603616;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15603617;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/mario_omescu/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=15603623;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15603624;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/mario_omescu/soccerdata/config/league_dict.json.                               

/Users/mario_omescu/Library/Mobile Documents/com~apple~CloudDocs/Sports Analytics/WhoScored Events Data


In [2]:
def get_chrome_major(path: str = CHROME_PATH) -> int:
    out = subprocess.check_output([path, "--version"], text=True)
    match = re.search(r"(\d+)\.", out)
    if not match:
        raise RuntimeError(f"Could not detect Chrome version from: {out}")
    return int(match.group(1))


def patch_soccerdata_chromedriver() -> None:
    def patched_init_webdriver(self):
        chrome_path = str(self.path_to_browser or CHROME_PATH)
        chrome_major = get_chrome_major(chrome_path)

        opts = uc.ChromeOptions()
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--start-maximized")

        return uc.Chrome(
            options=opts,
            version_main=chrome_major,
            browser_executable_path=chrome_path,
            headless=self.headless,
        )

    common.BaseSeleniumReader._init_webdriver = patched_init_webdriver


def patch_soccerdata_json_loader() -> None:
    def tolerant_json_load(fp, *args, **kwargs):
        content = fp.read()
        if isinstance(content, bytes):
            content = content.decode("utf-8", errors="ignore")

        content = content.strip()
        if content.startswith("<html"):
            start = content.find("{")
            end = content.rfind("}") + 1
            if start != -1 and end > start:
                content = content[start:end]

        return json.loads(content)

    json.load = tolerant_json_load


patch_soccerdata_chromedriver()
patch_soccerdata_json_loader()

print(f"SoccerData notebook patches applied. Chrome major: {get_chrome_major()}")

SoccerData notebook patches applied. Chrome major: 150


In [3]:
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

season_cfg = cfg["season"]
mongo_cfg = cfg["mongo"]
collections = mongo_cfg["collections"]

SEASON_YEAR = season_cfg["year"]
SEASON_SHORT = season_cfg["year_short"]
LEAGUE = season_cfg["league"]
COMPETITION_NAME = season_cfg["name"]
COMPETITION_COUNTRY = season_cfg["country"]

client = MongoClient(mongo_cfg["url"])
db = client[mongo_cfg["db"]]
collection_teams = db[collections["collection_teams"]]
collection_schedule = db[collections["collection_schedule"]]
collection_lineups = db[collections["collection_lineups"]]

available_teams = pd.DataFrame(collection_teams.find(
    {},
    {'_id':0,'ws_team_id': 1, 'ws_team_name': 1}))
schedule_docs = list(collection_schedule.find(
    {"season": SEASON_YEAR, "game_status": "finished"},
    {"_id": 0},
).sort("game_date", 1))


finished_games = pd.DataFrame(schedule_docs)
if finished_games.empty:
    raise ValueError(f"No finished games found in Mongo for season={SEASON_YEAR}")

finished_games["game_id"] = finished_games["game_id"].astype(int)
display(finished_games[["game_id", "home_team_name", "away_team_name", "week"]].head())
print(f"Finished games: {len(finished_games)}")

,game_id,home_team_name,away_team_name,week
0,1643039,Eintracht Frankfurt,Bayern Munich,1
1,1643046,Augsburg,Freiburg,1
2,1643045,Bochum,Mainz 05,1
3,1643043,Borussia M.Gladbach,Hoffenheim,1
4,1643041,Union Berlin,Hertha Berlin,1


Finished games: 306


In [5]:
ws = sd.WhoScored(
    leagues=LEAGUE,
    seasons=SEASON_YEAR,
    headless=True,
    path_to_browser=CHROME_PATH,
    no_cache=False,
)

def mongo_finished_schedule(self, force_cache=False):
    return pd.DataFrame({
        "league": LEAGUE,
        "season": SEASON_SHORT,
        "game": finished_games["game_id"].map(lambda x: f"match_{int(x)}"),
        "game_id": finished_games["game_id"].astype(int),
    })

ws.read_schedule = types.MethodType(mongo_finished_schedule, ws)

print("WhoScored reader ready")

[07/23/26 22:46:04] INFO     Saving cached data to /Users/mario_omescu/soccerdata/data/WhoScored     ]8;id=5614489;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=5614490;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

[07/23/26 22:46:05] INFO     patching driver executable /Users/mario_omescu/Library/Application      ]8;id=5614495;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py\patcher.py]8;;\:]8;id=5614496;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py#346\346]8;;\
                             Support/undetected_chromedriver/undetected_chromedriver                               

[07/23/26 22:46:07] INFO     setting properties for headless                                        ]8;id=5614501;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/__init__.py\__init__.py]8;;\:]8;id=5614502;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/__init__.py#493\493]8;;\

WhoScored reader ready


## Extract lineups

In [ ]:
LIMIT = None 
LIVE = False
RETRY_MISSING = True


def process_game_lineups(df_players: pd.DataFrame, game_id: int, available_teams: pd.DataFrame) -> pd.DataFrame:
    
    available_teams_df = available_teams.copy()
    available_teams_df.rename(columns={'ws_team_id': 'team_id', "ws_team_name": "team_name"}, inplace=True)

    df = df_players.copy()
    df = df.rename(columns={
        "is_starter": "starting_lineup",
    })

    df["game_id"] = int(game_id)
    df["season"] = SEASON_YEAR
    df["competition_name"] = COMPETITION_NAME
    df["competition_country"] = COMPETITION_COUNTRY

    for col in ["game_id", "team_id", "player_id", "minutes_played", "jersey_number"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    if "starting_lineup" in df.columns:
        df["starting_lineup"] = df["starting_lineup"].fillna(False).astype(bool)

    metadata_cols = [
        "game_id", "game_date", "week", "home_team_id", "home_team_name",
        "away_team_id", "away_team_name",
    ]
    metadata = finished_games[finished_games["game_id"].eq(int(game_id))][metadata_cols].drop_duplicates("game_id")
    df = df.merge(metadata, on="game_id", how="left")

    df["played"] = pd.to_numeric(df["minutes_played"], errors="coerce").fillna(0).gt(0)
    df["game_venue"] = df["team_id"].eq(df["home_team_id"]).map({True: "home", False: "away"})
    
    df = df.merge(available_teams_df, on='team_id', how='left')
    ordered = [
        "game_id", "game_date", "season", "week", "competition_name", "competition_country",
        "team_id", "team_name", "player_id", "player_name", "starting_lineup", "minutes_played",
        "jersey_number", "starting_position", "played", "game_venue"
    ]
    return df[[c for c in ordered if c in df.columns]]


lineup_frames = []
errors = []
game_ids = finished_games["game_id"].astype(int).tolist()
if LIMIT is not None:
    game_ids = game_ids[:LIMIT]

game_ids = [1643289]

for i, game_id in enumerate(game_ids, start=1):
    try:
        print(f"[{i}/{len(game_ids)}] Extracting players for game_id={game_id}")
        loader = ws.read_events(
            match_id=game_id,
            output_fmt="loader",
            retry_missing=RETRY_MISSING,
            live=LIVE,
            on_error="skip",
        )
        df_players = loader.players(game_id=game_id)
        lineup_frames.append(process_game_lineups(df_players, game_id, available_teams))
    except Exception as exc:
        logger.exception("Failed extracting players for game_id=%s", game_id)
        errors.append({"game_id": game_id, "error": repr(exc)})

df_lineups = pd.concat(lineup_frames, ignore_index=True) if lineup_frames else pd.DataFrame()
df_errors = pd.DataFrame(errors)

print(f"Lineup rows: {len(df_lineups)}")
print(f"Failed games: {len(df_errors)}")
display(df_lineups.head())
display(df_errors.head())

[1/1] Extracting players for game_id=1643289


[07/23/26 22:46:11] INFO     [1/1] Retrieving game with id=1643289                                 ]8;id=5614509;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5614510;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

                    ERROR    Failed extracting players for game_id=1643289                          ]8;id=5614517;file:///var/folders/jq/njqvkrp55b3g2fzq0k92__q80000gn/T/ipykernel_43484/313527250.py\313527250.py]8;;\:]8;id=5614518;file:///var/folders/jq/njqvkrp55b3g2fzq0k92__q80000gn/T/ipykernel_43484/313527250.py#68\68]8;;\
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/var/folders/jq/njqvkrp55b3g2fzq0k92__q80000gn/T/ipykernel_43484/3135                
                             27250.py", line 65, in <module>                                                       
                                 df_players = loader.players(game_id=game_id)                                      
                                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                      
                               File                                                                                
                             "/opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/pytho                
                             n3.11/site-packages/socceraction/data/opta/loader.py", line 427, in                   
                             players                                                                               
                                 _deepupdate(data, parser.extract_players())                                       
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^                                        
                               File                                                                                
                             "/opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/pytho                
                             n3.11/site-packages/socceraction/data/opta/parsers/whoscored.py", line                
                             162, in extract_players                                                               
                                 player_gamestats = self.extract_playergamestats()                                 
                                                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                 
                               File                                                                                
                             "/opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/pytho                
                             n3.11/site-packages/socceraction/data/opta/parsers/whoscored.py", line                
                             362, in extract_playergamestats                                                       
                                 for team in [self.root["home"], self.root["away"]]:                               
                                              ~~~~~~~~~^^^^^^^^                                                    
                             TypeError: 'NoneType' object is not subscriptable                                     

Lineup rows: 0
Failed games: 1


""


,game_id,error
0,1643289,"TypeError(""'NoneType' object is not subscripta..."


In [8]:
loader = ws.read_events(
            match_id=game_id,
            output_fmt="loader",
            retry_missing=RETRY_MISSING,
            live=LIVE,
            on_error="skip",
        )

[07/24/26 22:55:41] INFO     [1/1] Retrieving game with id=1643289                                 ]8;id=5614523;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5614524;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

In [11]:
ws = sd.WhoScored(leagues="GER-Bundesliga", seasons=2223)
print(ws.__doc__)


[07/24/26 22:57:22] INFO     Saving cached data to /Users/mario_omescu/soccerdata/data/WhoScored     ]8;id=5614529;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=5614530;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

[07/24/26 22:57:23] INFO     patching driver executable /Users/mario_omescu/Library/Application      ]8;id=5614535;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py\patcher.py]8;;\:]8;id=5614536;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py#346\346]8;;\
                             Support/undetected_chromedriver/undetected_chromedriver                               

Provides pd.DataFrames from data available at http://whoscored.com.

    Data will be downloaded as necessary and cached locally in
    ``~/soccerdata/data/WhoScored``.

    Parameters
    ----------
    leagues : string or iterable, optional
        IDs of Leagues to include.
    seasons : string, int or list, optional
        Seasons to include. Supports multiple formats.
        Examples: '16-17'; 2016; '2016-17'; [14, 15, 16]
    proxy : 'tor' or or dict or list(dict) or callable, optional
        Use a proxy to hide your IP address. Valid options are:
            - "tor": Uses the Tor network. Tor should be running in
              the background on port 9050.
            - str: The address of the proxy server to use.
            - list(str): A list of proxies to choose from. A different proxy will
              be selected from this list after failed requests, allowing rotating
              proxies.
            - callable: A function that returns a valid proxy. This function wil

In [12]:
loader = ws.read_events(output_fmt="loader")

[07/24/26 22:57:50] INFO     Retrieving calendar for GER-Bundesliga 2223                           ]8;id=5614542;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5614543;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#366\366]8;;\

                    INFO     [1/9] Retrieving fixtures for GER-Bundesliga 2223                     ]8;id=5614549;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5614550;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#397\397]8;;\

                    INFO     [2/9] Retrieving fixtures for GER-Bundesliga 2223                     ]8;id=5614555;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5614556;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#397\397]8;;\

                    INFO     [3/9] Retrieving fixtures for GER-Bundesliga 2223                     ]8;id=5614561;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5614562;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#397\397]8;;\

                    INFO     [4/9] Retrieving fixtures for GER-Bundesliga 2223                     ]8;id=5614567;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5614568;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#397\397]8;;\

...

                    INFO     [302/306] Retrieving game with id=1643308                             ]8;id=5616422;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5616423;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

                    INFO     [303/306] Retrieving game with id=1643190                             ]8;id=5616428;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5616429;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

                    INFO     [304/306] Retrieving game with id=1643132                             ]8;id=5616434;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5616435;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

                    INFO     [305/306] Retrieving game with id=1643194                             ]8;id=5616440;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5616441;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

                    INFO     [306/306] Retrieving game with id=1643184                             ]8;id=5616446;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=5616447;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

In [14]:
df_players = loader.players(game_id=1643229)

In [15]:
df_players

,game_id,team_id,player_id,player_name,is_starter,minutes_played,jersey_number,starting_position
0,1643229,7614,39187,Ørjan Nyland,True,98,13,GK
1,1643229,7614,372855,Mohamed Simakan,True,98,2,DR
2,1643229,7614,143600,Lukas Klostermann,True,98,16,DC
3,1643229,7614,104917,Willi Orbán,True,98,4,DC
4,1643229,7614,115161,Marcel Halstenberg,True,63,23,DL
5,1643229,7614,248010,Konrad Laimer,True,63,27,DMC
6,1643229,7614,298837,Amadou Haidara,True,48,8,DMC
7,1643229,7614,261212,Dani Olmo,True,98,7,AMC
8,1643229,7614,141716,Emil Forsberg,True,70,10,AMC
9,1643229,7614,300945,Christopher Nkunku,True,98,18,FW


In [ ]:
df_lineups['team_name'].value_counts()

In [7]:
SEASON_SHORT

'2223'